# GIC 2026 — QRC on QuEra Aquila (primary platform)

**Phase 3 hardware proof on the project's declared primary platform.** QuEra Aquila is a 256-atom **neutral-atom analog** machine; it runs a time-dependent Rydberg Hamiltonian, which is the *native* form of our transverse-field Ising reservoir.

Run order (cheapest first):
1. Build the reservoir + visualize the atom register.
2. **Local AHS simulator** smoke test (free) — confirm the encoding is input-sensitive.
3. Full forecast on the **free local simulator**.
4. **Check Aquila availability window.**
5. *(Spends credits)* small run on **real Aquila**.

> **Honest framing:** the claim is *the analog reservoir runs on real neutral-atom hardware, its Rydberg features survive device noise, and the forecast stays competitive with the best classical baselines.* Keep the Braket task ARNs as proof of hardware execution.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import numpy as np, matplotlib.pyplot as plt
from src.qrc.quera_reservoir import QueraReservoir
print('ready')

## 1. Build the reservoir and view the atom register

The fixed (seeded) atom geometry *is* the disordered reservoir coupling — van der Waals `C6/r⁶` set by where the atoms sit.

In [ ]:
res = QueraReservoir(n_atoms=5, geometry='random2d', seed=42)  # tuned winner (quera_tune.py)
print(res)

c = res._coords * 1e6  # micrometers
plt.figure(figsize=(4, 4))
plt.scatter(c[:, 0], c[:, 1], s=200, color='#7c3aed', edgecolor='k')
for i, (x, y) in enumerate(c):
    plt.annotate(str(i), (x, y), ha='center', va='center', color='white')
plt.xlabel('x (um)'); plt.ylabel('y (um)'); plt.gca().set_aspect('equal')
plt.title('Aquila register'); plt.grid(alpha=0.3); plt.show()

## 2. Local AHS simulator smoke test (free)

Confirm distinct inputs produce distinct reservoir features (spread should exceed the shot-noise floor).

In [ ]:
rng = np.random.RandomState(0)
X = rng.randn(5, res.n_atoms)
R = res.transform(X, device='local', shots=300)
print('feature matrix:', R.shape)
print('across-sample feature std :', round(float(R.std(0).mean()), 4), '(input sensitivity)')
Rc = res.transform(np.zeros((2, res.n_atoms)), device='local', shots=300)
print('identical-input feature MAE:', round(float(np.mean(np.abs(Rc[0]-Rc[1]))), 4), '(shot-noise floor)')

## 3. Full forecast on the free local AHS simulator

Trains the readout on free local-sim features and scores against Persistence/ESN. The local solver is ~2–3 s/program, so keep the caps modest here.

In [ ]:
from experiments.quera_aquila_qrc import run, _build_argparser

args = _build_argparser().parse_args([
    '--device', 'local', '--max-train', '150', '--max-test', '30', '--shots', '100',
])
run(args)

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/quera_aquila_summary.csv').round(5))
display(Image('results/quera_aquila_qrc.png'))

## 4. Check Aquila availability

Aquila is online only in scheduled windows. Submitting outside a window just queues the task. (Needs Braket access — provided in qBraid Lab.)

In [ ]:
from braket.aws import AwsDevice
from braket.devices import Devices

aquila = AwsDevice(Devices.QuEra.Aquila)
print('status      :', aquila.status)
print('is_available:', aquila.is_available)
print('execution windows:')
for w in aquila.properties.service.executionWindows:
    print('  ', w)

## 5. Real QuEra Aquila run — spends credits

Run this only during an availability window. `--allow-qpu` is the required safety latch. Cost ≈ `tasks × ($0.30 + shots × $0.01)` (e.g. 40 tasks × 100 shots ≈ $52).

In [ ]:
# UNCOMMENT to run on real QuEra Aquila (consumes credits):
# args = _build_argparser().parse_args([
#     '--device', 'aquila', '--max-train', '300', '--max-test', '40',
#     '--shots', '100', '--allow-qpu',
# ])
# run(args)
#
# display(pd.read_csv('results/quera_aquila_summary.csv').round(5))
# display(Image('results/quera_aquila_qrc.png'))